# Week 03. Hidden Pairs

**Как пользоваться**
1. В ячейке «Настройки» выбери `SCORE_MODE`, модели и число триалов `N_TRIALS`.
2. Run All. Для каждой модели считаются CV ROC-AUC (5 фолдов на train) и ROC-AUC на validation.
3. Лучшие параметры печатаются после прогона: скопируй их в `DEFAULT_PARAMS`,
   верни `N_TRIALS` = 1 и сделай финальный Restart → Run All.

**Сабмиты:** `submission_1_<модель>.csv` — вариант с лучшим validation ROC-AUC,
`submission_2_<модель>.csv` — второй; `final_submission.csv` = копия первого.

In [1]:
# ============================ НАСТРОЙКИ ============================
SEED = 20260916

# Чем Optuna выбирает параметры:
#   "valid" — ROC-AUC на validation.csv
#   "cv"    — OOF ROC-AUC по фолдам train
#   "mean"  — среднее двух
SCORE_MODE = "mean"

# Какие модели запускать (обязательные: dt, rf, hgb)
MODELS_TO_RUN = ["dt", "rf", "hgb", "nystroem_ridge", "svm", "svm_w", "knn", "mlp", "pt_mlp"]
# Какие из них смешивать в ансамбль
ENSEMBLE_MEMBERS = ["nystroem_ridge", "svm", "svm_w", "knn", "mlp", "pt_mlp", "hgb"]

# Число триалов на модель.
#   1  — запускаются только DEFAULT_PARAMS, перебора нет (финальный прогон);
#   >1 — первый триал = DEFAULT_PARAMS, остальные Optuna выбирает из space_<модель>.
N_TRIALS = {name: 1 for name in MODELS_TO_RUN}
#N_TRIALS["svm_w"] = 30
#N_TRIALS["knn"] = 50

ENSEMBLE_TRIALS = 300
N_FOLDS = 5

In [2]:
from pathlib import Path
import os
import platform
import random
import time

import numpy as np
import optuna
import pandas as pd
import scipy
import sklearn
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin
from sklearn.calibration import CalibratedClassifierCV
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.validation import check_is_fitted

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
optuna.logging.set_verbosity(optuna.logging.WARNING)

for name, module in [("NumPy", np), ("pandas", pd), ("SciPy", scipy), ("scikit-learn", sklearn),
                     ("Optuna", optuna), ("PyTorch", torch)]:
    print(f"{name}: {module.__version__}")
print("Python:", platform.python_version())

/home/anechka/3 курс/введение в мо/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NumPy: 2.5.2
pandas: 3.0.5
SciPy: 1.18.1
scikit-learn: 1.9.0
Optuna: 5.0.0
PyTorch: 2.14.0+cu130
Python: 3.14.4


In [3]:
# Ищем папку с данными: локально или на Kaggle
candidates = [Path("."), Path("week-03-hidden-pairs"), *sorted(Path("/kaggle/input").glob("*"))]
DATA_DIR = next(p for p in candidates if (p / "train.csv").exists())
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
print("DATA_DIR:", DATA_DIR)

train = pd.read_csv(DATA_DIR / "train.csv")
validation = pd.read_csv(DATA_DIR / "validation.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

feature_columns = [c for c in train.columns if c not in ("row_id", "target")]
assert feature_columns == [c for c in validation.columns if c not in ("row_id", "target")]
assert feature_columns == [c for c in test.columns if c != "row_id"]
assert test["row_id"].equals(sample_submission["row_id"])

X_train = train[feature_columns].to_numpy(dtype=np.float32)
y_train = train["target"].to_numpy(dtype=np.int8)
X_valid = validation[feature_columns].to_numpy(dtype=np.float32)
y_valid = validation["target"].to_numpy(dtype=np.int8)
X_test = test[feature_columns].to_numpy(dtype=np.float32)

folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(X_train, y_train))

print("train:", X_train.shape, "validation:", X_valid.shape, "test:", X_test.shape)
print("positive rate train / valid:", y_train.mean().round(3), y_valid.mean().round(3))

DATA_DIR: week-03-hidden-pairs
train: (2520, 512) validation: (560, 512) test: (1116, 512)
positive rate train / valid: 0.5 0.5


## Модели

Каждая модель описывается тремя вещами:
- `build(params, seed)` — собирает sklearn `Pipeline`;
- `space(trial)` — пространство поиска для Optuna;
- `DEFAULT_PARAMS[name]` — стартовые (лучшие известные) параметры, всегда проверяются первым триалом.

Общий препроцессинг (`preprocess_steps`) управляется параметрами:
- `use_log1p` — `log1p` признаков;
- `pca_n_components` — PCA до указанного числа компонент;
- `pca_power` — перевзвешивание PCA: компонента умножается на `explained_variance ** pca_power`.
  `0` — обычный PCA, `> 0` — главные компоненты усиливаются относительно хвостовых.

In [4]:
class WeightedPCA(BaseEstimator, TransformerMixin):
    """PCA, после которого каждая компонента умножается на explained_variance ** power."""

    def __init__(self, n_components=None, power=0.0, random_state=None):
        self.n_components = n_components
        self.power = power
        self.random_state = random_state

    def fit(self, X, y=None):
        self.pca_ = PCA(n_components=self.n_components, random_state=self.random_state).fit(X)
        self.scale_ = self.pca_.explained_variance_ ** self.power
        return self

    def transform(self, X):
        return self.pca_.transform(X) * self.scale_


def preprocess_steps(params, seed, scale=True):
    steps = []
    if params.get("use_log1p", False):
        steps.append(("log1p", FunctionTransformer(np.log1p)))
    if scale:
        steps.append(("scaler", StandardScaler()))
    if params.get("pca_n_components"):
        steps.append(("pca", WeightedPCA(n_components=params["pca_n_components"],
                                         power=params.get("pca_power", 0.0), random_state=seed)))
    return steps


class TorchMLPClassifier(BaseEstimator, ClassifierMixin):
    """Небольшой MLP на PyTorch в sklearn-обёртке. Всегда CPU — ради воспроизводимости."""

    def __init__(self, hidden_dim=128, n_layers=2, dropout=0.2, lr=1e-3,
                 epochs=50, batch_size=128, seed=SEED):
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.dropout = dropout
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.seed = seed

    def _build(self, input_dim):
        layers, in_dim = [], input_dim
        for _ in range(self.n_layers):
            layers += [nn.Linear(in_dim, self.hidden_dim), nn.BatchNorm1d(self.hidden_dim),
                       nn.ReLU(), nn.Dropout(self.dropout)]
            in_dim = self.hidden_dim
        layers.append(nn.Linear(in_dim, 1))
        return nn.Sequential(*layers)

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        torch.manual_seed(self.seed)
        self.model_ = self._build(X.shape[1])
        optimizer = torch.optim.AdamW(self.model_.parameters(), lr=self.lr, weight_decay=1e-2)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.epochs)
        criterion = nn.BCEWithLogitsLoss()
        dataset = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32))
        loader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True,
                            generator=torch.Generator().manual_seed(self.seed))
        self.model_.train()
        for _ in range(self.epochs):
            for batch_X, batch_y in loader:
                optimizer.zero_grad()
                criterion(self.model_(batch_X).squeeze(-1), batch_y).backward()
                optimizer.step()
            scheduler.step()
        return self

    def predict_proba(self, X):
        check_is_fitted(self)
        self.model_.eval()
        with torch.no_grad():
            p = torch.sigmoid(self.model_(torch.tensor(X, dtype=torch.float32)).squeeze(-1)).numpy()
        return np.column_stack([1 - p, p])

In [5]:
MODELS = {}
DEFAULT_PARAMS = {}


def register(name, build, space, default):
    MODELS[name] = (build, space)
    DEFAULT_PARAMS[name] = default


# ---------- Decision tree (обязательная) ----------
def build_dt(p, seed):
    return Pipeline(preprocess_steps(p, seed, scale=False) + [("model", DecisionTreeClassifier(
        max_depth=p["max_depth"], min_samples_leaf=p["min_samples_leaf"], criterion=p["criterion"],
        max_features=p["max_features"], ccp_alpha=p["ccp_alpha"], random_state=seed))])

def space_dt(t):
    return {
        "max_depth": t.suggest_int("max_depth", 2, 16),
        "min_samples_leaf": t.suggest_int("min_samples_leaf", 1, 80, log=True),
        "criterion": t.suggest_categorical("criterion", ["gini", "entropy"]),
        "max_features": t.suggest_float("max_features", 0.1, 1.0),
        "ccp_alpha": t.suggest_float("ccp_alpha", 1e-6, 1e-2, log=True),
    }

register("dt", build_dt, space_dt,
         {"max_depth": 6, "min_samples_leaf": 20, "criterion": "gini",
                "max_features": 1.0, "ccp_alpha": 1e-6})


# ---------- Random forest (обязательная) ----------
def build_rf(p, seed):
    return Pipeline(preprocess_steps(p, seed, scale=False) + [("model", RandomForestClassifier(
        n_estimators=p["n_estimators"], max_depth=p["max_depth"], min_samples_leaf=p["min_samples_leaf"],
        max_features=p["max_features"], max_samples=p["max_samples"], n_jobs=-1, random_state=seed))])

def space_rf(t):
    return {
        "n_estimators": t.suggest_int("n_estimators", 200, 1000, step=100),
        "max_depth": t.suggest_int("max_depth", 6, 40),
        "min_samples_leaf": t.suggest_int("min_samples_leaf", 1, 10),
        "max_features": t.suggest_float("max_features", 0.02, 0.4, log=True),
        "max_samples": t.suggest_float("max_samples", 0.5, 1.0),
    }

register("rf", build_rf, space_rf,
         {"n_estimators": 500, "max_depth": 20, "min_samples_leaf": 2,
                "max_features": 0.05, "max_samples": 1.0})


# ---------- Gradient boosting (обязательная) ----------
def build_hgb(p, seed):
    return Pipeline(preprocess_steps(p, seed, scale=False) + [("model", HistGradientBoostingClassifier(
        learning_rate=p["learning_rate"], max_iter=p["max_iter"], max_leaf_nodes=p["max_leaf_nodes"],
        min_samples_leaf=p["min_samples_leaf"], l2_regularization=p["l2_regularization"],
        max_features=p["max_features"], early_stopping=False, random_state=seed))])

def space_hgb(t):
    return {
        "learning_rate": t.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "max_iter": t.suggest_int("max_iter", 100, 800, step=50),
        "max_leaf_nodes": t.suggest_int("max_leaf_nodes", 4, 63),
        "min_samples_leaf": t.suggest_int("min_samples_leaf", 5, 100, log=True),
        "l2_regularization": t.suggest_float("l2_regularization", 1e-3, 10.0, log=True),
        "max_features": t.suggest_float("max_features", 0.05, 1.0),
    }

register("hgb", build_hgb, space_hgb,
         {"learning_rate": 0.05, "max_iter": 400, "max_leaf_nodes": 15, "min_samples_leaf": 20,
                 "l2_regularization": 1.0, "max_features": 0.3})


# ---------- Extra trees ----------
def build_et(p, seed):
    return Pipeline(preprocess_steps(p, seed, scale=False) + [("model", ExtraTreesClassifier(
        n_estimators=p["n_estimators"], max_depth=p["max_depth"], min_samples_leaf=p["min_samples_leaf"],
        max_features=p["max_features"], criterion=p["criterion"], n_jobs=-1, random_state=seed))])

def space_et(t):
    return {
        "n_estimators": t.suggest_int("n_estimators", 300, 1000, step=100),
        "max_depth": t.suggest_int("max_depth", 10, 50),
        "min_samples_leaf": t.suggest_int("min_samples_leaf", 1, 5),
        "max_features": t.suggest_float("max_features", 0.05, 0.6, log=True),
        "criterion": t.suggest_categorical("criterion", ["gini", "entropy"]),
    }

register("et", build_et, space_et,
         {"n_estimators": 900, "max_depth": 24, "min_samples_leaf": 3,
                "max_features": 0.46, "criterion": "gini"})


# ---------- Logistic regression ----------
def build_lr(p, seed):
    return Pipeline(preprocess_steps(p, seed) + [("model", LogisticRegression(
        C=p["C"], max_iter=2000, random_state=seed))])

def space_lr(t):
    return {
        "use_log1p": t.suggest_categorical("use_log1p", [True, False]),
        "C": t.suggest_float("C", 1e-4, 1e1, log=True),
    }

register("lr", build_lr, space_lr,
         {"use_log1p": True, "C": 0.0044})


# ---------- Nystroem + Ridge ----------
def build_nystroem_ridge(p, seed):
    return Pipeline(preprocess_steps(p, seed) + [
        ("nystroem", Nystroem(gamma=p["gamma"], n_components=p["nystroem_n_components"], random_state=seed)),
        ("model", CalibratedClassifierCV(RidgeClassifier(alpha=p["alpha"]), ensemble=False)),
    ])

def space_nystroem_ridge(t):
    return {
        "use_log1p": t.suggest_categorical("use_log1p", [True, False]),
        "nystroem_n_components": t.suggest_int("nystroem_n_components", 256, 2048, step=64),
        "gamma": t.suggest_float("gamma", 5e-4, 1e-2, log=True),
        "alpha": t.suggest_float("alpha", 1e-2, 1e2, log=True),
    }

register("nystroem_ridge", build_nystroem_ridge, space_nystroem_ridge,
         {"use_log1p": True, "nystroem_n_components": 1024,
                            "gamma": 0.001794456600464692, "alpha": 0.25075529962608184})


# ---------- SVM (RBF) ----------
def build_svm(p, seed):
    return Pipeline(preprocess_steps(p, seed) + [
        ("model", CalibratedClassifierCV(SVC(C=p["C"], gamma=p["gamma"]), ensemble=False)),
    ])

def space_svm(t):
    return {
        "use_log1p": t.suggest_categorical("use_log1p", [True, False]),
        "pca_n_components": t.suggest_int("pca_n_components", 64, 512, step=16),
        "C": t.suggest_float("C", 0.3, 20.0, log=True),
        "gamma": t.suggest_float("gamma", 5e-4, 1e-2, log=True),
    }

register("svm", build_svm, space_svm,
         {"use_log1p": True, "pca_n_components": 400,
                 "C": 2.5201026001524154, "gamma": 0.0030261093912262887})


# ---------- SVM (RBF) на перевзвешенном PCA ----------
def space_svm_w(t):
    return {
        "use_log1p": t.suggest_categorical("use_log1p", [True, False]),
        "pca_n_components": t.suggest_int("pca_n_components", 256, 512, step=16),
        "pca_power": t.suggest_float("pca_power", 0.0, 0.5),
        "C": t.suggest_float("C", 1.0, 30.0, log=True),
        "gamma": t.suggest_float("gamma", 5e-4, 1e-2, log=True),
    }

register("svm_w", build_svm, space_svm_w,
         {"use_log1p": True, "pca_n_components": 464, "pca_power": 0.1875, "C": 5.0, "gamma": 1 / 464})


# ---------- kNN ----------
def build_knn(p, seed):
    return Pipeline(preprocess_steps(p, seed) + [("model", KNeighborsClassifier(
        n_neighbors=p["n_neighbors"], weights=p["weights"], p=p["p"], n_jobs=-1))])

def space_knn(t):
    use_pca = t.suggest_categorical("use_pca", [True, False])
    return {
        "use_log1p": t.suggest_categorical("use_log1p", [True, False]),
        "pca_n_components": t.suggest_int("pca_n_components", 16, 256, step=16) if use_pca else None,
        "pca_power": t.suggest_float("pca_power", 0.0, 0.5) if use_pca else 0.0,
        "n_neighbors": t.suggest_int("n_neighbors", 5, 101, step=2),
        "weights": t.suggest_categorical("weights", ["uniform", "distance"]),
        "p": t.suggest_int("p", 1, 2),
    }

register("knn", build_knn, space_knn,
         {"use_pca": True, "use_log1p": True, "pca_n_components": 32, "pca_power": 0.0,
                 "n_neighbors": 17, "weights": "distance", "p": 2})


# ---------- sklearn MLP ----------
def build_mlp(p, seed):
    return Pipeline(preprocess_steps(p, seed) + [("model", MLPClassifier(
        hidden_layer_sizes=tuple(p[f"n_units_l{i}"] for i in range(p["n_layers"])),
        activation=p["activation"], alpha=p["alpha"], learning_rate_init=p["learning_rate_init"],
        max_iter=300, early_stopping=True, n_iter_no_change=15, random_state=seed))])

def space_mlp(t):
    n_layers = t.suggest_int("n_layers", 1, 3)
    p = {
        "use_log1p": t.suggest_categorical("use_log1p", [True, False]),
        "n_layers": n_layers,
        "activation": t.suggest_categorical("activation", ["relu", "tanh"]),
        "alpha": t.suggest_float("alpha", 1e-5, 1e-1, log=True),
        "learning_rate_init": t.suggest_float("learning_rate_init", 1e-4, 1e-2, log=True),
    }
    for i in range(n_layers):
        p[f"n_units_l{i}"] = t.suggest_int(f"n_units_l{i}", 32, 256, step=32)
    return p

register("mlp", build_mlp, space_mlp,
         {"use_log1p": False, "n_layers": 2, "n_units_l0": 64, "n_units_l1": 64,
                 "activation": "relu", "alpha": 0.0010114589792259566,
                 "learning_rate_init": 0.0010957766093919455})


# ---------- PyTorch MLP ----------
def build_pt_mlp(p, seed):
    return Pipeline(preprocess_steps(p, seed) + [("model", TorchMLPClassifier(
        hidden_dim=p["hidden_dim"], n_layers=p["n_layers"], dropout=p["dropout"], lr=p["lr"],
        epochs=p["epochs"], batch_size=p["batch_size"], seed=seed))])

def space_pt_mlp(t):
    use_pca = t.suggest_categorical("use_pca", [True, False])
    return {
        "use_log1p": t.suggest_categorical("use_log1p", [True, False]),
        "pca_n_components": t.suggest_int("pca_n_components", 32, 256, step=16) if use_pca else None,
        "pca_power": t.suggest_float("pca_power", 0.0, 0.5) if use_pca else 0.0,
        "n_layers": t.suggest_int("n_layers", 1, 3),
        "hidden_dim": t.suggest_int("hidden_dim", 64, 512, step=64),
        "dropout": t.suggest_float("dropout", 0.1, 0.4),
        "lr": t.suggest_float("lr", 1e-4, 5e-3, log=True),
        "epochs": t.suggest_int("epochs", 20, 80),
        "batch_size": t.suggest_categorical("batch_size", [64, 128, 256]),
    }

register("pt_mlp", build_pt_mlp, space_pt_mlp,
         {"use_pca": True, "use_log1p": False, "pca_n_components": 48, "pca_power": 0.0, "n_layers": 3,
                    "hidden_dim": 512, "dropout": 0.22594958370617121, "lr": 0.0001818223247471324,
                    "epochs": 37, "batch_size": 256})
DEFAULT_PARAMS["knn"] = {'use_pca': True, 'use_log1p': True, 'pca_n_components': 32, 'pca_power': 0.06977688911426896, 'n_neighbors': 19, 'weights': 'distance', 'p': 2}

## Оценка одной конфигурации

Для каждого набора параметров:
1. **CV**: 5 раз обучаем на 4/5 train, предсказываем 1/5 → OOF ROC-AUC;
2. **Validation**: обучаем на всём train, предсказываем validation и test.

Модель, обученная на train, сразу даёт предсказания для test — повторно обучать ничего не нужно.

In [6]:
def evaluate(name, params):
    build, _ = MODELS[name]
    start = time.perf_counter()

    oof = np.zeros(len(y_train))
    for tr_idx, va_idx in folds:
        model = build(params, SEED).fit(X_train[tr_idx], y_train[tr_idx])
        oof[va_idx] = model.predict_proba(X_train[va_idx])[:, 1]

    model = build(params, SEED).fit(X_train, y_train)
    valid_pred = model.predict_proba(X_valid)[:, 1]
    test_pred = model.predict_proba(X_test)[:, 1]

    return {
        "cv_auc": roc_auc_score(y_train, oof),
        "valid_auc": roc_auc_score(y_valid, valid_pred),
        "oof": oof, "valid_pred": valid_pred, "test_pred": test_pred,
        "runtime_s": time.perf_counter() - start,
    }


def select_score(cv_auc, valid_auc):
    return {"valid": valid_auc, "cv": cv_auc, "mean": (cv_auc + valid_auc) / 2}[SCORE_MODE]


def make_objective(name):
    _, space = MODELS[name]

    def objective(trial):
        res = evaluate(name, space(trial))
        for key, value in res.items():
            trial.set_user_attr(key, value)
        return select_score(res["cv_auc"], res["valid_auc"])
    return objective

## Прогон моделей

In [7]:
studies = {}
for name in MODELS_TO_RUN:
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.enqueue_trial(DEFAULT_PARAMS[name])
    study.optimize(make_objective(name), n_trials=N_TRIALS[name])
    studies[name] = study

    best = study.best_trial
    print(f"{name:15s} cv={best.user_attrs['cv_auc']:.4f}  valid={best.user_attrs['valid_auc']:.4f}  "
          f"({best.user_attrs['runtime_s']:.0f}s/trial, {len(study.trials)} trials)")

dt              cv=0.7443  valid=0.7856  (5s/trial, 1 trials)
rf              cv=0.8916  valid=0.8978  (17s/trial, 1 trials)
hgb             cv=0.9051  valid=0.9166  (29s/trial, 1 trials)
nystroem_ridge  cv=0.9212  valid=0.9344  (17s/trial, 1 trials)
svm             cv=0.9255  valid=0.9470  (38s/trial, 1 trials)
svm_w           cv=0.9253  valid=0.9548  (30s/trial, 1 trials)
knn             cv=0.9077  valid=0.9266  (3s/trial, 1 trials)
mlp             cv=0.9103  valid=0.9227  (9s/trial, 1 trials)
pt_mlp          cv=0.9162  valid=0.9359  (38s/trial, 1 trials)


In [8]:
# Лучшие параметры — копируй их в DEFAULT_PARAMS после перебора
for name, study in studies.items():
    print(f'DEFAULT_PARAMS["{name}"] = {study.best_params}')

DEFAULT_PARAMS["dt"] = {'max_depth': 6, 'min_samples_leaf': 20, 'criterion': 'gini', 'max_features': 1.0, 'ccp_alpha': 1e-06}
DEFAULT_PARAMS["rf"] = {'n_estimators': 500, 'max_depth': 20, 'min_samples_leaf': 2, 'max_features': 0.05, 'max_samples': 1.0}
DEFAULT_PARAMS["hgb"] = {'learning_rate': 0.05, 'max_iter': 400, 'max_leaf_nodes': 15, 'min_samples_leaf': 20, 'l2_regularization': 1.0, 'max_features': 0.3}
DEFAULT_PARAMS["nystroem_ridge"] = {'use_log1p': True, 'nystroem_n_components': 1024, 'gamma': 0.001794456600464692, 'alpha': 0.25075529962608184}
DEFAULT_PARAMS["svm"] = {'use_log1p': True, 'pca_n_components': 400, 'C': 2.5201026001524154, 'gamma': 0.0030261093912262887}
DEFAULT_PARAMS["svm_w"] = {'use_log1p': True, 'pca_n_components': 464, 'pca_power': 0.1875, 'C': 5.0, 'gamma': 0.0021551724137931034}
DEFAULT_PARAMS["knn"] = {'use_pca': True, 'use_log1p': True, 'pca_n_components': 32, 'pca_power': 0.06977688911426896, 'n_neighbors': 19, 'weights': 'distance', 'p': 2}
DEFAULT_PARAM

## Ансамбль

In [9]:
members = [m for m in ENSEMBLE_MEMBERS if m in studies]
oof = {m: studies[m].best_trial.user_attrs["oof"] for m in members}
val = {m: studies[m].best_trial.user_attrs["valid_pred"] for m in members}
tst = {m: studies[m].best_trial.user_attrs["test_pred"] for m in members}


def blend(preds, w):
    return sum(preds[m] * w[m] for m in members) / sum(w.values())


def ensemble_objective(trial):
    w = {m: trial.suggest_float(f"w_{m}", 0.0, 1.0) for m in members}
    if sum(w.values()) == 0:
        return 0.0
    return select_score(roc_auc_score(y_train, blend(oof, w)), roc_auc_score(y_valid, blend(val, w)))


ens_study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
ens_study.enqueue_trial({f"w_{m}": 1.0 for m in members})
ens_study.optimize(ensemble_objective, n_trials=ENSEMBLE_TRIALS)

raw = {m: ens_study.best_params[f"w_{m}"] for m in members}
weights = {m: w / sum(raw.values()) for m, w in raw.items()}
equal = {m: 1.0 for m in members}

print("Веса:", {m: round(w, 3) for m, w in weights.items()})
print(f"Равные веса:     cv={roc_auc_score(y_train, blend(oof, equal)):.4f}  "
      f"valid={roc_auc_score(y_valid, blend(val, equal)):.4f}")
print(f"Подобранные:     cv={roc_auc_score(y_train, blend(oof, weights)):.4f}  "
      f"valid={roc_auc_score(y_valid, blend(val, weights)):.4f}")

Веса: {'nystroem_ridge': 0.004, 'svm': 0.33, 'svm_w': 0.425, 'knn': 0.16, 'mlp': 0.065, 'pt_mlp': 0.014, 'hgb': 0.002}
Равные веса:     cv=0.9285  valid=0.9470
Подобранные:     cv=0.9285  valid=0.9531


## Таблица экспериментов

`features` — какой препроцессинг использовался. Таблица сохраняется в `results/experiments.csv`.

In [10]:
def describe_features(p):
    parts = ["log1p" if p.get("use_log1p") else "raw"]
    if p.get("pca_n_components"):
        parts.append(f"PCA{p['pca_n_components']}")
    if p.get("pca_power"):
        parts.append(f"power{p['pca_power']:.3g}")
    return " + ".join(parts) + " (512 feats)"


rows = []
for name, study in studies.items():
    b = study.best_trial
    rows.append({"model": name, "features": describe_features(b.params), "cv_roc_auc": b.user_attrs["cv_auc"],
                 "valid_roc_auc": b.user_attrs["valid_auc"], "runtime_s": b.user_attrs["runtime_s"],
                 "seed": SEED, "n_trials": len(study.trials)})
rows.append({"model": "ensemble(" + ",".join(members) + ")", "features": "blend of members",
             "cv_roc_auc": roc_auc_score(y_train, blend(oof, weights)),
             "valid_roc_auc": roc_auc_score(y_valid, blend(val, weights)),
             "runtime_s": sum(studies[m].best_trial.user_attrs["runtime_s"] for m in members),
             "seed": SEED, "n_trials": ENSEMBLE_TRIALS})

experiments = pd.DataFrame(rows).sort_values("valid_roc_auc", ascending=False)
experiments.to_csv(RESULTS_DIR / "experiments.csv", index=False)
experiments.round(4)

,model,features,cv_roc_auc,valid_roc_auc,runtime_s,seed,n_trials
5,svm_w,log1p + PCA464 + power0.188 (512 feats),0.9253,0.9548,29.9704,20260916,1
9,"ensemble(nystroem_ridge,svm,svm_w,knn,mlp,pt_m...",blend of members,0.9285,0.9531,164.4554,20260916,300
4,svm,log1p + PCA400 (512 feats),0.9255,0.9470,38.3579,20260916,1
8,pt_mlp,raw + PCA48 (512 feats),0.9162,0.9359,38.3348,20260916,1
3,nystroem_ridge,log1p (512 feats),0.9212,0.9344,16.5911,20260916,1
6,knn,log1p + PCA32 + power0.0698 (512 feats),0.9077,0.9266,3.0487,20260916,1
7,mlp,raw (512 feats),0.9103,0.9227,8.7413,20260916,1
2,hgb,raw (512 feats),0.9051,0.9166,29.4113,20260916,1
1,rf,raw (512 feats),0.8916,0.8978,16.6077,20260916,1
0,dt,raw (512 feats),0.7443,0.7856,5.4353,20260916,1


## Сабмиты

Кандидаты: лучшая одиночная модель по validation и ансамбль.
Нумерация по validation ROC-AUC: `submission_1_*` — лучший, `submission_2_*` — второй.

In [11]:
best_single = max(studies, key=lambda m: studies[m].best_trial.user_attrs["valid_auc"])
candidates = {
    best_single: (studies[best_single].best_trial.user_attrs["valid_auc"],
                  studies[best_single].best_trial.user_attrs["test_pred"]),
    "ensemble": (roc_auc_score(y_valid, blend(val, weights)), blend(tst, weights)),
}
ranked = sorted(candidates.items(), key=lambda item: item[1][0], reverse=True)

for rank, (name, (auc, pred)) in enumerate(ranked, start=1):
    submission = sample_submission[["row_id"]].copy()
    submission["target"] = np.clip(pred, 0.0, 1.0)

    assert list(submission.columns) == list(sample_submission.columns)
    assert submission["row_id"].equals(sample_submission["row_id"])
    assert submission["target"].between(0, 1).all() and not submission.isna().any().any()

    filename = f"submission_{rank}_{name}.csv"
    submission.to_csv(filename, index=False)
    if rank == 1:
        submission.to_csv("final_submission.csv", index=False)
    print(f"{filename:32s} valid ROC-AUC = {auc:.4f}")
print("final_submission.csv =", f"submission_1_{ranked[0][0]}.csv")

submission_1_svm_w.csv           valid ROC-AUC = 0.9548
submission_2_ensemble.csv        valid ROC-AUC = 0.9531
final_submission.csv = submission_1_svm_w.csv
